# MedCore Document Intelligence Benchmark — Track B: Table Extraction
### Models: Docling + TableFormer (Accurate) vs. TATR (Table Transformer)

This notebook evaluates and compares the performance of two state-of-the-art table structure recognition models:
1. **Docling + TableFormer** (Accurate Mode)
2. **TATR (Table Transformer)** structure-recognition-v1.1-all

Both models are evaluated against the verified ground truth on the 4-page MedCore product catalogue.

In [1]:
# Setup imports and sys.path
import os
import sys
import zipfile
import json
import io
import shutil
import itertools
import time
import numpy as np
import pandas as pd
from PIL import Image
import img2pdf
import fitz  # PyMuPDF
import torch
import torchvision.transforms as T
import pytesseract
from transformers import AutoImageProcessor, TableTransformerConfig, TableTransformerForObjectDetection
from huggingface_hub import hf_hub_download
from PIL import ImageOps
from html import escape
from lxml import etree
from apted import APTED, Config
import jiwer as _jiwer

# Ensure grits.py is importable
sys.path.append(".")
sys.path.append("/kaggle/working")

# Set Tesseract command for Windows local testing
if os.name == 'nt':
    tess_path = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
    if os.path.exists(tess_path):
        pytesseract.pytesseract.tesseract_cmd = tess_path

# Target directories
output_dir = "/kaggle/working" if os.path.exists("/kaggle") else "./"
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Output directory: /kaggle/working


## Step 1: PDF Reconstruction / Preparation
In the Kaggle environment, the input PDF may be packed as a ZIP container containing rendered images. Locally, it is a standard PDF. We handle both cases seamlessly:
- If standard PDF: copy it directly and render page images via `fitz` for TATR cropping.
- If ZIP container: extract page JPEGs in page order, compile them losslessly using `img2pdf` to `/kaggle/working/medcore_real.pdf`, and cache the rendered page images.

In [2]:
# Find PDF using fallbacks
pdf_candidates = [
    "/kaggle/input/medical-document-layout-annotator/MedCore_Catalogue_v2.pdf",
    "../input/medical-document-layout-annotator/MedCore_Catalogue_v2.pdf",
    "MedCore_Catalogue_v2.pdf",
    "./MedCore_Catalogue_v2.pdf",
    "d:/antigravity/benchmarking/MedCore_Catalogue_v2.pdf"
]
pdf_path = None
for c in pdf_candidates:
    if os.path.exists(c):
        pdf_path = c
        break
if pdf_path is None:
    # Try finding any pdf in current directory
    pdf_files = [f for f in os.listdir(".") if f.endswith(".pdf")]
    if pdf_files:
        pdf_path = pdf_files[0]
assert pdf_path is not None, "Could not find MedCore_Catalogue_v2.pdf"
print(f"Found input PDF: {pdf_path}")

output_pdf_path = os.path.join(output_dir, "medcore_real.pdf")
page_images = {}
page_dims = {}

# Check if input is a ZIP container
is_zip = zipfile.is_zipfile(pdf_path)
print(f"Is input a ZIP file? {is_zip}")

if is_zip:
    print("Processing as ZIP container...")
    with zipfile.ZipFile(pdf_path) as z:
        names = z.namelist()
        manifest_name = next((n for n in names if n.endswith("manifest.json")), None)
        assert manifest_name is not None, "manifest.json not found in ZIP!"
        
        manifest = json.loads(z.read(manifest_name))
        pages = sorted(manifest["pages"], key=lambda p: p["page_number"])
        
        image_paths_in_order = []
        temp_dir = os.path.join(output_dir, "temp_pages")
        os.makedirs(temp_dir, exist_ok=True)
        
        for p in pages:
            page_num = p["page_number"]
            img_path_in_zip = p["image"]["path"]
            img_bytes = z.read(img_path_in_zip)
            
            temp_img_path = os.path.join(temp_dir, f"{page_num}.jpg")
            with open(temp_img_path, "wb") as f:
                f.write(img_bytes)
            image_paths_in_order.append(temp_img_path)
            
            page_images[page_num] = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            print(f"Cached page {page_num}: size={page_images[page_num].size}")

        # Convert to PDF losslessly via img2pdf
        pdf_bytes = img2pdf.convert(image_paths_in_order)
        with open(output_pdf_path, "wb") as f:
            f.write(pdf_bytes)
        print(f"PDF losslessly written to {output_pdf_path}")
else:
    print("Processing as standard PDF file...")
    shutil.copy(pdf_path, output_pdf_path)
    print(f"PDF copied directly to {output_pdf_path}")
    
    # Open and render pages to PIL images for caching
    doc = fitz.open(output_pdf_path)
    for i, page in enumerate(doc, start=1):
        pix = page.get_pixmap(matrix=fitz.Matrix(1, 1))
        img_bytes = pix.tobytes("png")
        page_images[i] = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        print(f"Cached page {i}: size={page_images[i].size}")

# Verify via PyMuPDF
doc = fitz.open(output_pdf_path)
print(f"Verified PDF page count: {len(doc)}")
assert len(doc) == 4, f"Expected 4 pages, got {len(doc)}"

for i, page in enumerate(doc, start=1):
    rect = page.rect
    width_pts, height_pts = rect.width, rect.height
    page_dims[i] = (width_pts, height_pts)
    print(f"Page {i} dimensions (pts): width={width_pts:.2f}, height={height_pts:.2f}")
    
    pix = page.get_pixmap()
    assert pix is not None and len(pix.samples) > 0, f"Page {i} rendered empty pixmap!"
    print(f"Page {i} successfully rendered to image.")

print("Step 1 completed successfully.")

Found input PDF: MedCore_Catalogue_v2.pdf
Is input a ZIP file? False
Processing as standard PDF file...
PDF copied directly to /kaggle/working\medcore_real.pdf
Cached page 1: size=(596, 842)


Cached page 2: size=(596, 842)
Cached page 3: size=(596, 842)
Cached page 4: size=(596, 842)
Verified PDF page count: 4
Page 1 dimensions (pts): width=595.28, height=841.89
Page 1 successfully rendered to image.
Page 2 dimensions (pts): width=595.28, height=841.89
Page 2 successfully rendered to image.
Page 3 dimensions (pts): width=595.28, height=841.89
Page 3 successfully rendered to image.
Page 4 dimensions (pts): width=595.28, height=841.89


Page 4 successfully rendered to image.
Step 1 completed successfully.


## Step 2: Load and Index Table Ground Truth
We load `MedCore_GT_v2_with_tables.json` and extract the target tables:
- `e0043` (Page 3 comparison table, 14 rows x 4 columns)
- `e0074` (Page 4 ServicePlus plans, 4 rows x 3 columns)

For each target table, we compute:
1. Pixel-space bounding box (top-left origin, for TATR cropping)
2. PDF-point bounding box (bottom-left origin, for Docling IoU matching)

In [3]:
# Find GT file using fallbacks
gt_candidates = [
    "/kaggle/input/medical-document-layout-annotator/MedCore_GT_v2_with_tables.json",
    "../input/medical-document-layout-annotator/MedCore_GT_v2_with_tables.json",
    "MedCore_GT_v2_with_tables.json",
    "./MedCore_GT_v2_with_tables.json",
    "d:/antigravity/benchmarking/MedCore_GT_v2_with_tables.json"
]
gt_with_tables_path = None
for c in gt_candidates:
    if os.path.exists(c):
        gt_with_tables_path = c
        break
assert gt_with_tables_path is not None, "Could not find MedCore_GT_v2_with_tables.json"
print(f"Found GT file: {gt_with_tables_path}")

with open(gt_with_tables_path, "r", encoding="utf-8") as f:
    gt_full = json.load(f)

table_gt = {}
for page in gt_full["pages"]:
    page_num = page["page"]
    for el in page["elements"]:
        if el.get("gt_table_status") == "programmatic_verified":
            tid = el["id"]
            bn = el["bbox_norm"]
            
            # Pixel-space bbox (top-left origin, for PIL cropping in TATR)
            W, H = page_images[page_num].size
            bbox_pixel = {
                "x0": bn["x"] * W,
                "y0": bn["y"] * H,
                "x1": (bn["x"] + bn["w"]) * W,
                "y1": (bn["y"] + bn["h"]) * H
            }
            
            # PDF-point bbox (origin bottom-left, for Docling IoU matching)
            p_w, p_h = page_dims[page_num]
            bbox_pdf = {
                "l": bn["x"] * p_w,
                "b": (1.0 - bn["y"] - bn["h"]) * p_h,
                "r": (bn["x"] + bn["w"]) * p_w,
                "t": (1.0 - bn["y"]) * p_h
            }
            
            table_gt[tid] = {
                "page": page_num,
                "bbox_norm": bn,
                "bbox_pixel": bbox_pixel,
                "bbox_pdf": bbox_pdf,
                "html": el["gt_table_html"],
                "markdown": el["gt_table_markdown"],
                "cells": el["gt_table_cells"],
                "meta": el["gt_table_meta"]
            }

print(f"Loaded {len(table_gt)} target tables:")
for tid, data in table_gt.items():
    print(f"Table {tid}: Page {data['page']}, Grid Shape={data['meta']['n_rows']}x{data['meta']['n_cols']}")

Found GT file: MedCore_GT_v2_with_tables.json
Loaded 2 target tables:
Table e0043: Page 3, Grid Shape=14x4
Table e0074: Page 4, Grid Shape=4x3


## Step 3: TEDS Metric Porting & Verification
We port the Table Edit Distance (TEDS) metric verbatim from the Track A repository, utilizing `APTED` tree-edit distance and `jiwer` character error rate. We include 3 assertions (known-answer sanity checks) to verify correctness:
1. Identical HTML tables must return TEDS = 1.0.
2. Changing a single cell content must yield TEDS < 1.0 and TEDS-Struct = 1.0.
3. Deleting an entire row must yield both TEDS < 1.0 and TEDS-Struct < 1.0.

In [4]:
# TEDS Metric Implementation
class _TableTree:
    def __init__(self, tag, text=None, children=None):
        self.tag = tag
        self.text = text
        self.children = children or []

def _html_to_tree(html_str):
    root_el = etree.fromstring(f"<root>{html_str}</root>", parser=etree.HTMLParser())
    table_el = root_el.find(".//table")

    def build(el):
        tag = el.tag
        if tag in ("td", "th"):
            text = "".join(el.itertext()).strip()
            return _TableTree(tag, text=text, children=[])
        children = [build(c) for c in el if c.tag in ("thead", "tbody", "tr", "td", "th")]
        return _TableTree(tag, text=None, children=children)

    return build(table_el)

def _strip_content(t):
    if t.tag in ("td", "th"):
        t.text = None
    for c in t.children:
        _strip_content(c)
    return t

def _count_nodes(t):
    return 1 + sum(_count_nodes(c) for c in t.children)

class _TEDSConfig(Config):
    def rename(self, n1, n2):
        if n1.tag != n2.tag:
            return 1.0
        if n1.tag in ("td", "th"):
            t1, t2 = n1.text or "", n2.text or ""
            if t1 == t2:
                return 0.0
            try:
                return min(1.0, _jiwer.cer(t1, t2)) if t1 else 1.0
            except Exception:
                return 1.0
        return 0.0

    def children(self, node):
        return node.children

    def get_label(self, node):
        return node.tag

def teds_score(html_gt, html_pred, structure_only=False):
    t1, t2 = _html_to_tree(html_gt), _html_to_tree(html_pred)
    if structure_only:
        t1, t2 = _strip_content(t1), _strip_content(t2)
    ed = APTED(t1, t2, _TEDSConfig()).compute_edit_distance()
    n = max(_count_nodes(t1), _count_nodes(t2))
    return 1.0 - (ed / n if n else 0.0)

# --- Sanity checks ---
_gt = "<table><thead><tr><th>A</th><th>B</th></tr></thead><tbody><tr><td>1</td><td>2</td></tr></tbody></table>"
_identical = _gt
_one_wrong = _gt.replace(">2<", ">9<")
_missing_row = "<table><thead><tr><th>A</th><th>B</th></tr></thead><tbody></tbody></table>"

# Assertion 1: Identical tables -> TEDS == 1.0
assert abs(teds_score(_gt, _identical) - 1.0) < 1e-9, "Assertion 1 failed"
print("Assertion 1 passed (Identical -> TEDS 1.0)")

# Assertion 2: One wrong cell -> TEDS < 1.0 and TEDS-Struct == 1.0
t_val = teds_score(_gt, _one_wrong)
ts_val = teds_score(_gt, _one_wrong, structure_only=True)
assert t_val < 1.0, "Assertion 2a failed"
assert abs(ts_val - 1.0) < 1e-9, "Assertion 2b failed"
print(f"Assertion 2 passed (One wrong cell -> TEDS={t_val:.4f}, TEDS-Struct={ts_val:.4f})")

# Assertion 3: Missing row -> TEDS < 1.0 and TEDS-Struct < 1.0
t_miss = teds_score(_gt, _missing_row)
ts_miss = teds_score(_gt, _missing_row, structure_only=True)
assert t_miss < 1.0, "Assertion 3a failed"
assert ts_miss < 1.0, "Assertion 3b failed"
print(f"Assertion 3 passed (Missing row -> TEDS={t_miss:.4f}, TEDS-Struct={ts_miss:.4f})")

print("All TEDS assertions passed successfully.")

Assertion 1 passed (Identical -> TEDS 1.0)
Assertion 2 passed (One wrong cell -> TEDS=0.8889, TEDS-Struct=1.0000)
Assertion 3 passed (Missing row -> TEDS=0.6667, TEDS-Struct=0.6667)
All TEDS assertions passed successfully.


## Step 4: Docling + TableFormer (Accurate Mode) Pipeline
We run the `docling` DocumentConverter on the compiled `medcore_real.pdf`. The pipeline is configured in `ACCURATE` mode with OCR enabled.

Since Docling runs on the full PDF without pre-cropping (as required), we match the detected tables to GT tables using Intersection over Union (IoU) on their bounding boxes in PDF bottom-left point coordinates.
- If best IoU < 0.3, we assign TEDS = 0.
- Otherwise, we export the table to HTML (`export_to_html(result.document)`), score TEDS / TEDS-Struct, and write the HTML outputs to `/kaggle/working/`.

In [5]:
print("=== Running Docling + TableFormer Pipeline ===")
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode

# Initialize Docling in Accurate mode
options = PdfPipelineOptions(do_table_structure=True, do_ocr=True)
options.table_structure_options.do_cell_matching = True
options.table_structure_options.mode = TableFormerMode.ACCURATE

converter = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=options)})

start_time = time.time()
result = converter.convert(output_pdf_path)
elapsed = time.time() - start_time
print(f"Docling conversion completed in {elapsed:.2f}s")
print(f"Total detected tables: {len(result.document.tables)}")

for i, detection in enumerate(result.document.tables):
    page_no = detection.prov[0].page_no
    bbox = detection.prov[0].bbox
    print(f"Detection {i}: page={page_no}, bbox={bbox}")

# Table matching using IoU
docling_results = {}
for tid, gt_table in table_gt.items():
    print(f"\nMatching GT table {tid} (page {gt_table['page']})...")
    max_iou = 0.0
    matched_detection = None
    
    gt_bbox = gt_table["bbox_pdf"]
    gt_page = gt_table["page"]
    
    for i, detection in enumerate(result.document.tables):
        det_page = detection.prov[0].page_no
        if det_page != gt_page:
            continue
            
        bbox = detection.prov[0].bbox
        # Compute IoU in bottom-left PDF space
        l_int = max(gt_bbox["l"], bbox.l)
        b_int = max(gt_bbox["b"], bbox.b)
        r_int = min(gt_bbox["r"], bbox.r)
        t_int = min(gt_bbox["t"], bbox.t)
        
        w_int = max(0.0, r_int - l_int)
        h_int = max(0.0, t_int - b_int)
        area_int = w_int * h_int
        
        area_gt = (gt_bbox["r"] - gt_bbox["l"]) * (gt_bbox["t"] - gt_bbox["b"])
        area_det = (bbox.r - bbox.l) * (bbox.t - bbox.b)
        
        union = area_gt + area_det - area_int
        iou = area_int / union if union > 0 else 0.0
        print(f"  vs Detection {i}: IoU={iou:.4f}")
        
        if iou > max_iou:
            max_iou = iou
            matched_detection = detection
            
    print(f"  Best Match IoU: {max_iou:.4f}")
    if max_iou < 0.3:
        print(f"Docling failed to detect table {tid}")
        docling_results[tid] = {
            "html": "",
            "iou": max_iou,
            "teds": 0.0,
            "teds_struct": 0.0,
            "rows": 0,
            "cols": 0
        }
    else:
        pred_html = matched_detection.export_to_html(result.document)
        # Compute TEDS
        teds = teds_score(gt_table["html"], pred_html)
        teds_struct = teds_score(gt_table["html"], pred_html, structure_only=True)
        
        n_rows = matched_detection.data.num_rows
        n_cols = matched_detection.data.num_cols
        
        docling_results[tid] = {
            "html": pred_html,
            "iou": max_iou,
            "teds": teds,
            "teds_struct": teds_struct,
            "rows": n_rows,
            "cols": n_cols
        }
        print(f"  TEDS: {teds:.4f}, TEDS-Struct: {teds_struct:.4f}")
        
        # Save HTML
        html_out_path = os.path.join(output_dir, f"docling_table_{tid}.html")
        with open(html_out_path, "w", encoding="utf-8") as f:
            f.write(pred_html)
        print(f"  Saved to {html_out_path}")

print("Step 4 completed successfully.")

=== Running Docling + TableFormer Pipeline ===


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights:  86%|████████▌ | 664/770 [00:00<00:00, 6597.86it/s]

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 6610.95it/s]

C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Docling conversion completed in 34.74s
Total detected tables: 4
Detection 0: page=3, bbox=l=307.08892822265625 t=491.9986572265625 r=563.6422119140625 b=450.25543212890625 coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>
Detection 1: page=3, bbox=l=26.512163162231445 t=333.8572998046875 r=567.59765625 b=29.0948486328125 coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>
Detection 2: page=4, bbox=l=212.77163696289062 t=629.5218811035156 r=382.8992919921875 b=534.8120422363281 coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>
Detection 3: page=4, bbox=l=398.1481628417969 t=550.1833190917969 r=562.1209106445312 b=496.4786376953125 coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>

Matching GT table e0043 (page 3)...
  vs Detection 0: IoU=0.0000
  vs Detection 1: IoU=0.9151
  Best Match IoU: 0.9151


  TEDS: 0.7368, TEDS-Struct: 0.7368
  Saved to /kaggle/working\docling_table_e0043.html

Matching GT table e0074 (page 4)...
  vs Detection 2: IoU=0.8459
  vs Detection 3: IoU=0.0000
  Best Match IoU: 0.8459
  TEDS: 0.7222, TEDS-Struct: 0.7222
  Saved to /kaggle/working\docling_table_e0074.html
Step 4 completed successfully.


## Step 5: TATR (Table Transformer) Pipeline
We implement the custom pre-cropped TATR pipeline:
1. For each target table, crop the region from the page image using `bbox_pixel`. Pad the crop by 4px and upscale by x2 to improve OCR downstream.
2. Apply `MaxResize(1000)` and standard PyTorch normalization.
3. Perform a forward pass through the TATR structure model (`microsoft/table-transformer-structure-recognition-v1.1-all`). We work around the strict dataclass validation issue of newer Hugging Face Hub releases by manually patching the loaded model configuration.
4. Scale predictions back to the upscaled crop space.
5. Apply the conditional fallback: if row count is far from GT (off by >3), retry inference once at threshold 0.3.
6. Intersect row boxes and column boxes to reconstruct the grid; crop each intersection cell and OCR using Tesseract `--psm 6`.
7. Mark rows as headers if they have `IoU > 0.5` with detected column headers.
8. Reconstruct HTML and score against ground truth.

In [6]:
print("=== Running TATR Pipeline ===")

# Patch TATR config to avoid strict dataclass validation error
processor = AutoImageProcessor.from_pretrained("microsoft/table-transformer-structure-recognition-v1.1-all")

config_path = hf_hub_download(repo_id="microsoft/table-transformer-structure-recognition-v1.1-all", filename="config.json")
config_dict = json.load(open(config_path))
config_dict["dilation"] = False
config = TableTransformerConfig.from_dict(config_dict)

model = TableTransformerForObjectDetection.from_pretrained("microsoft/table-transformer-structure-recognition-v1.1-all", config=config)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
print(f"TATR loaded on device: {device}")

# Helper for IoU between two 2D boxes
def compute_box_iou(box1, box2):
    l_int = max(box1[0], box2[0])
    b_int = max(box1[1], box2[1])
    r_int = min(box1[2], box2[2])
    t_int = min(box1[3], box2[3])
    
    w_int = max(0.0, r_int - l_int)
    h_int = max(0.0, t_int - b_int)
    area_int = w_int * h_int
    
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union = area1 + area2 - area_int
    return area_int / union if union > 0 else 0.0

# MaxResize Transform
class MaxResize:
    def __init__(self, max_size=1000):
        self.max_size = max_size
    def __call__(self, img):
        w, h = img.size
        scale = self.max_size / max(w, h)
        new_w = int(w * scale)
        new_h = int(h * scale)
        return img.resize((new_w, new_h), Image.BILINEAR), scale

resize_transform = MaxResize(1000)
normalize_transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def ocr_cell(cell_img):
    padded = ImageOps.expand(cell_img, border=10, fill="white")
    gray = padded.convert("L")
    return pytesseract.image_to_string(gray, config="--psm 6").strip()

tatr_results = {}

for tid, gt_table in table_gt.items():
    print(f"\nProcessing TATR for table {tid}...")
    page_num = gt_table["page"]
    bbox_pixel = gt_table["bbox_pixel"]
    
    # Crop table image with pad 4px
    W, H = page_images[page_num].size
    x0 = max(0, int(bbox_pixel["x0"]) - 4)
    y0 = max(0, int(bbox_pixel["y0"]) - 4)
    x1 = min(W, int(bbox_pixel["x1"]) + 4)
    y1 = min(H, int(bbox_pixel["y1"]) + 4)
    crop = page_images[page_num].crop((x0, y0, x1, y1))
    
    # Upscale x2
    crop_upscaled = crop.resize((crop.width * 2, crop.height * 2), Image.LANCZOS)
    
    # Apply MaxResize & normalization
    resized_crop, scale = resize_transform(crop_upscaled)
    tensor_crop = normalize_transform(resized_crop)
    res_h, res_w = resized_crop.height, resized_crop.width
    
    # Forward pass
    with torch.no_grad():
        outputs = model(tensor_crop.unsqueeze(0).to(device))
    
    target_sizes = torch.tensor([[res_h, res_w]]).to(device)
    
    # Try threshold 0.5 first
    threshold = 0.5
    results = processor.post_process_object_detection(outputs, threshold=threshold, target_sizes=target_sizes)[0]
    
    row_boxes = [box.cpu().numpy() / scale for label, box in zip(results["labels"], results["boxes"]) if model.config.id2label[label.item()] == "table row"]
    gt_rows = gt_table["meta"]["n_rows"]
    
    # Fallback retry at threshold 0.3 if row count is far from GT
    if abs(len(row_boxes) - gt_rows) > 3:
        print(f"  Row count {len(row_boxes)} is far from GT {gt_rows}. Retrying with threshold 0.3...")
        threshold = 0.3
        results = processor.post_process_object_detection(outputs, threshold=threshold, target_sizes=target_sizes)[0]
    
    # Extract final scaled boxes
    row_boxes = []
    col_boxes = []
    header_boxes = []
    
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box = box.cpu().numpy()
        label_name = model.config.id2label[label.item()]
        orig_box = box / scale
        if label_name == "table row":
            row_boxes.append(orig_box)
        elif label_name == "table column":
            col_boxes.append(orig_box)
        elif label_name == "table column header":
            header_boxes.append(orig_box)
            
    print(f"  Detections: rows={len(row_boxes)}, cols={len(col_boxes)}, headers={len(header_boxes)}")
    
    # Sort boxes
    row_boxes.sort(key=lambda b: (b[1] + b[3]) / 2) # top-to-bottom
    col_boxes.sort(key=lambda b: (b[0] + b[2]) / 2) # left-to-right
    
    # Grid cell intersection and OCR
    grid = []
    n_rows = len(row_boxes)
    n_cols = len(col_boxes)
    
    for r in range(n_rows):
        grid_row = []
        r_box = row_boxes[r]
        for c in range(n_cols):
            c_box = col_boxes[c]
            
            # Intersection box
            x0_int = max(r_box[0], c_box[0])
            y0_int = max(r_box[1], c_box[1])
            x1_int = min(r_box[2], c_box[2])
            y1_int = min(r_box[3], c_box[3])
            
            w_int = max(0.0, x1_int - x0_int)
            h_int = max(0.0, y1_int - y0_int)
            area_int = w_int * h_int
            
            if area_int < 25.0:
                grid_row.append("")
            else:
                cell_crop = crop_upscaled.crop((x0_int, y0_int, x1_int, y1_int))
                cell_text = ocr_cell(cell_crop)
                grid_row.append(cell_text)
        grid.append(grid_row)
        
    # Header row matching
    row_is_header = []
    for r_box in row_boxes:
        is_header = False
        for h_box in header_boxes:
            if compute_box_iou(r_box, h_box) > 0.5:
                is_header = True
                break
        row_is_header.append(is_header)
        
    # Reconstruct HTML
    html_lines = ["<table>"]
    has_headers = any(row_is_header)
    if has_headers:
        html_lines.append("  <thead>")
        for r in range(n_rows):
            if row_is_header[r]:
                html_lines.append("    <tr>" + "".join(f"<th>{escape(grid[r][c])}</th>" for c in range(n_cols)) + "</tr>")
        html_lines.append("  </thead>")
        
    html_lines.append("  <tbody>")
    for r in range(n_rows):
        if not row_is_header[r] or not has_headers:
            cells_html = []
            for c in range(n_cols):
                is_col_header = (c == 0 and gt_table["meta"]["has_header_col"])
                tag = "th" if is_col_header else "td"
                cells_html.append(f"<{tag}>{escape(grid[r][c])}</{tag}>")
            html_lines.append("    <tr>" + "".join(cells_html) + "</tr>")
    html_lines.append("  </tbody>")
    html_lines.append("</table>")
    
    pred_html = "\n".join(html_lines)
    
    teds = teds_score(gt_table["html"], pred_html)
    teds_struct = teds_score(gt_table["html"], pred_html, structure_only=True)
    
    tatr_results[tid] = {
        "html": pred_html,
        "grid": grid,
        "teds": teds,
        "teds_struct": teds_struct,
        "rows": n_rows,
        "cols": n_cols,
        "threshold": threshold
    }
    print(f"  TEDS: {teds:.4f}, TEDS-Struct: {teds_struct:.4f}")
    
    # Save HTML
    html_out_path = os.path.join(output_dir, f"tatr_table_{tid}.html")
    with open(html_out_path, "w", encoding="utf-8") as f:
        f.write(pred_html)
    print(f"  Saved to {html_out_path}")

=== Running TATR Pipeline ===


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

Loading weights:  91%|█████████ | 333/367 [00:00<00:00, 3284.58it/s]

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 3350.66it/s]

TATR loaded on device: cpu

Processing TATR for table e0043...


  Detections: rows=14, cols=4, headers=1


  TEDS: 0.8285, TEDS-Struct: 1.0000
  Saved to /kaggle/working\tatr_table_e0043.html

Processing TATR for table e0074...


  Detections: rows=4, cols=3, headers=1


  TEDS: 0.6602, TEDS-Struct: 0.7368
  Saved to /kaggle/working\tatr_table_e0074.html


## Step 6 & 7: GriTS-Top Metric, Cell-Level F1 & Results Aggregation
We import `grits.py` to evaluate structural cell topology (GriTS-Top). To guarantee PyMuPDF type compatibility with `grits.py` when running on recent library environments, we monkey-patch `grits.iou` to convert NumPy arrays into standard Python lists of floats before passing them to PyMuPDF's `Rect` constructor.

We also implement a custom cell-level Precision, Recall, and F1 scorer that maps ground truth cells to predicted cells using the optimal row/column alignment sequence computed by GriTS.

In [7]:
print("=== Running GriTS and Cell-F1 ===")
import grits

# Monkey patch grits.iou to convert numpy arrays or any sequence to list of floats for fitz compatibility
def patched_iou(bbox1, bbox2):
    if hasattr(bbox1, "tolist"):
        bbox1 = bbox1.tolist()
    if hasattr(bbox2, "tolist"):
        bbox2 = bbox2.tolist()
    bbox1 = [float(x) for x in bbox1]
    bbox2 = [float(x) for x in bbox2]
    
    intersection = fitz.Rect(bbox1).intersect(fitz.Rect(bbox2))
    union = fitz.Rect(bbox1).include_rect(fitz.Rect(bbox2))
    
    union_area = union.get_area()
    if union_area > 0:
        return intersection.get_area() / union.get_area()
    return 0.0

grits.iou = patched_iou

def compute_cell_level_metrics(true_html, pred_html):
    true_cells = grits.html_to_cells(true_html)
    pred_cells = grits.html_to_cells(pred_html)
    
    true_grid = np.array(grits.cells_to_grid(true_cells, key='cell_text'), dtype=object)
    pred_grid = np.array(grits.cells_to_grid(pred_cells, key='cell_text'), dtype=object)
    
    # Align rows/cols using grits-top alignment
    true_topology_grid = np.array(grits.cells_to_relspan_grid(true_cells))
    pred_topology_grid = np.array(grits.cells_to_relspan_grid(pred_cells))
    
    pre_computed_rewards = {}
    for trow, tcol, prow, pcol in itertools.product(range(true_topology_grid.shape[0]),
                                                    range(true_topology_grid.shape[1]),
                                                    range(pred_topology_grid.shape[0]),
                                                    range(pred_topology_grid.shape[1])):
        pre_computed_rewards[(trow, tcol, prow, pcol)] = grits.iou(true_topology_grid[trow, tcol], pred_topology_grid[prow, pcol])
        
    true_row_nums, pred_row_nums, _ = grits.align_2d_outer(true_topology_grid.shape[:2],
                                                           pred_topology_grid.shape[:2],
                                                           pre_computed_rewards)
                                                           
    transpose_rewards = {(tcol, trow, pcol, prow): val for (trow, tcol, prow, pcol), val in pre_computed_rewards.items()}
    true_column_nums, pred_column_nums, _ = grits.align_2d_outer(true_topology_grid.shape[:2][::-1],
                                                                 pred_topology_grid.shape[:2][::-1],
                                                                 transpose_rewards)
                                                                 
    def norm(s):
        import re
        return re.sub(r"\s+", " ", (s or "").strip().lower())
        
    tp = 0
    aligned_rows = list(zip(true_row_nums, pred_row_nums))
    aligned_cols = list(zip(true_column_nums, pred_column_nums))
    
    for tr, pr in aligned_rows:
        for tc, pc in aligned_cols:
            t_val = true_grid[tr][tc]
            p_val = pred_grid[pr][pc]
            if norm(t_val) == norm(p_val) and norm(t_val) != "":
                tp += 1
                
    num_true = sum(1 for r in range(true_grid.shape[0]) for c in range(true_grid.shape[1]) if norm(true_grid[r][c]) != "")
    num_pred = sum(1 for r in range(pred_grid.shape[0]) for c in range(pred_grid.shape[1]) if norm(pred_grid[r][c]) != "")
    
    precision = tp / num_pred if num_pred > 0 else 1.0
    recall = tp / num_true if num_true > 0 else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return precision, recall, f1

records = []
for tid, gt_table in table_gt.items():
    # Model 1: Docling
    doc_res = docling_results[tid]
    d_metrics = grits.grits_from_html(gt_table["html"], doc_res["html"])
    d_prec, d_rec, d_f1 = compute_cell_level_metrics(gt_table["html"], doc_res["html"])
    
    records.append({
        "table_id": tid,
        "table_title": gt_table["meta"]["title"],
        "model": "docling",
        "TEDS": doc_res["teds"],
        "TEDS_Struct": doc_res["teds_struct"],
        "GriTS_Top": d_metrics["grits_top"],
        "Cell_Precision": d_prec,
        "Cell_Recall": d_rec,
        "Cell_F1": d_f1,
        "detected_rows": doc_res["rows"],
        "gt_rows": gt_table["meta"]["n_rows"],
        "detected_cols": doc_res["cols"],
        "gt_cols": gt_table["meta"]["n_cols"],
        "detection_iou": doc_res["iou"],
        "confidence_threshold": None
    })
    
    # Model 2: TATR
    tatr_res = tatr_results[tid]
    t_metrics = grits.grits_from_html(gt_table["html"], tatr_res["html"])
    t_prec, t_rec, t_f1 = compute_cell_level_metrics(gt_table["html"], tatr_res["html"])
    
    records.append({
        "table_id": tid,
        "table_title": gt_table["meta"]["title"],
        "model": "tatr",
        "TEDS": tatr_res["teds"],
        "TEDS_Struct": tatr_res["teds_struct"],
        "GriTS_Top": t_metrics["grits_top"],
        "Cell_Precision": t_prec,
        "Cell_Recall": t_rec,
        "Cell_F1": t_f1,
        "detected_rows": tatr_res["rows"],
        "gt_rows": gt_table["meta"]["n_rows"],
        "detected_cols": tatr_res["cols"],
        "gt_cols": gt_table["meta"]["n_cols"],
        "detection_iou": None,
        "confidence_threshold": tatr_res["threshold"]
    })

results_df = pd.DataFrame(records)
print("\n=== Per-Table Results ===")
print(results_df.to_string())

# Assemble Leaderboard
leaderboard = results_df.groupby("model")[["TEDS", "TEDS_Struct", "GriTS_Top", "Cell_F1"]].mean().reset_index()
print("\n=== Leaderboard ===")
print(leaderboard.to_string())

# Save CSVs
results_df.to_csv(os.path.join(output_dir, "table_benchmark_per_table_results.csv"), index=False)
leaderboard.to_csv(os.path.join(output_dir, "table_benchmark_leaderboard.csv"), index=False)
print("CSVs saved successfully.")

=== Running GriTS and Cell-F1 ===



=== Per-Table Results ===
  table_id                                   table_title    model      TEDS  TEDS_Struct  GriTS_Top  Cell_Precision  Cell_Recall   Cell_F1  detected_rows  gt_rows  detected_cols  gt_cols  detection_iou  confidence_threshold
0    e0043  Patient Monitor Series -- Feature Comparison  docling  0.736842     0.736842   0.965517        0.933333     1.000000  0.965517             15       14              4        4       0.915132                   NaN
1    e0043  Patient Monitor Series -- Feature Comparison     tatr  0.828525     1.000000   1.000000        0.442308     0.410714  0.425926             14       14              4        4            NaN                   0.5
2    e0074                             ServicePlus Plans  docling  0.722222     0.722222   1.000000        1.000000     1.000000  1.000000              4        4              3        3       0.845915                   NaN
3    e0074                             ServicePlus Plans     tatr  0.660196  

## Step 8: Discussion Metrics
We compute the specific performance differences requested:
1. The structural-vs-content degradation (`TEDS_Struct - TEDS`) for both models on both tables.
2. The direct TEDS difference between Docling and TATR on table `e0074` (which lacks a header row).

In [8]:
print("=== Running Discussion Metrics ===")
for _, row in results_df.iterrows():
    diff = row["TEDS_Struct"] - row["TEDS"]
    print(f"Table {row['table_id']} - Model {row['model']}: TEDS_Struct - TEDS = {diff:.4f}")

# Direct model comparison for e0074 (no header row)
doc_e0074_teds = results_df[(results_df["table_id"] == "e0074") & (results_df["model"] == "docling")]["TEDS"].values[0]
tatr_e0074_teds = results_df[(results_df["table_id"] == "e0074") & (results_df["model"] == "tatr")]["TEDS"].values[0]
print(f"e0074 comparison (Docling TEDS - TATR TEDS) = {doc_e0074_teds - tatr_e0074_teds:.4f}")
print("Step 8 completed successfully.")

=== Running Discussion Metrics ===
Table e0043 - Model docling: TEDS_Struct - TEDS = 0.0000
Table e0043 - Model tatr: TEDS_Struct - TEDS = 0.1715
Table e0074 - Model docling: TEDS_Struct - TEDS = 0.0000
Table e0074 - Model tatr: TEDS_Struct - TEDS = 0.0766
e0074 comparison (Docling TEDS - TATR TEDS) = 0.0620
Step 8 completed successfully.
